# Banco de pruebas

Aqui se prueba una consulta antes de llevarla al ETL. El notebook no forma
parte del proceso automatico: es el sitio donde se mira el resultado, se
comprueba el grano y se corrige. Cuando la consulta esta bien, se guarda en
`queries/` y se anade a `CONSULTAS` en `src/olist_ETL.py`.

Ejecutad las celdas en orden.

In [ ]:
import sys
from pathlib import Path

# El notebook vive en notebooks/, asi que hay que anadir la raiz al path
# para poder importar src/.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
from src.olist_ETL import crear_engine, extraer, comprobar_grano, CONSULTAS

engine = crear_engine()
print("Conexion lista. Consultas registradas:", list(CONSULTAS))

## 1. Ejecutar una consulta existente

Cambiad `NOMBRE` por cualquiera de las registradas.

In [ ]:
NOMBRE = "clientes_actividad"

df = extraer(engine, NOMBRE)
print(df.shape)
df.head()

## 2. Comprobar el grano

Esta es la comprobacion que el ETL hace en cada ejecucion. Si el numero de
filas no coincide con el de valores distintos de la clave, algun JOIN esta
multiplicando y el CSV saldria inflado.

In [ ]:
clave = CONSULTAS[NOMBRE]

print(f"filas: {len(df)}")
print(f"{clave} distintos: {df[clave].nunique()}")

comprobar_grano(df, clave, NOMBRE)
print("Grano correcto.")

## 3. Probar SQL suelto

Para iterar sin tocar todavia ningun fichero `.sql`.

In [ ]:
consulta = """
SELECT order_status, COUNT(*) AS pedidos
FROM orders
GROUP BY order_status
ORDER BY pedidos DESC
"""

pd.read_sql(consulta, engine)

## 4. Dejarla lista para el ETL

1. Guardad la consulta en `queries/<nombre>.sql`, con el grano declarado en la primera linea.
2. Anadid `"<nombre>": "<columna_de_grano>"` al diccionario `CONSULTAS` de `src/olist_ETL.py`.
3. Ejecutad `python main.py` desde la raiz.

No hay que tocar nada mas: el ETL recorre `CONSULTAS` y genera un CSV por entrada.